# ___Species coverage in FRED subsets___
----------------------------------------------

In [1]:
!python --version

Python 3.13.11


The system cannot find the path specified.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [17]:
#--------------------------
# CONSTANTS
#--------------------------

COLLABORATION_AXES = ["F00679", "F00727"] # RD, SRL
PLANT_TAXONOMY_ACCEPTED_COLUMNS = [ "F01286", "F01287", "F01289", "F01290" ] # genus name, specific epithet, family & order
BINOMINAL = [ "F01286", "F01287"] # genus name & specific epithet

In [43]:
fred = pd.read_csv(r"../../data/chapter2/FRED/FRED3_Entire_Database_2021.csv", low_memory=False, header=0, skiprows=range(1, 10), encoding="latin1") # FRED v3
meta = pd.read_csv(r"../../data/chapter2/FRED/FRED3_Column_Definitions_2021.csv", index_col="column_id") # metadata for FRED v3
# taxonlookup .csv data
lookup = pd.read_csv(r"../../data/chapter2/plant_lookup.csv", low_memory=False, encoding="latin1", usecols=["genus", "apweb.family", "order", "group"], index_col="genus").rename(mapper={"apweb.family": "family"}, axis=1) 

# let's use FungalRoot geneus level mycorrhizal state recommendations to estimate the state diversity in FRED v3
# recommended mycorrhizal status for plant genera, based on at least 67% consistency of species diagnosis for a given mycorrhizal type
fungalroot_genus_level_states = pd.read_excel(r"../../data/chapter2/FungalRoot/nph16569-sup-0002-tabless1-s4.xlsx", sheet_name="Table S2", skiprows=range(2))
fungalroot_genus_level_states = {gen: state for (gen, state) in zip(fungalroot_genus_level_states.Genus.str.strip(), fungalroot_genus_level_states.loc[:, "Mycorrhizal type"])}

In [80]:
meta.loc[COLLABORATION_AXES + PLANT_TAXONOMY_ACCEPTED_COLUMNS, ["name", "units", "definition"]]

,name,units,definition
column_id,,,
F00679,Root diameter,mm,Diameter of roots observed.
F00727,Specific root length (SRL),m/g,Length of roots divided by root mass.
F01286,Plant taxonomy_Accepted genus_TPL,NaN,Genus of plant according to The Plant List: Th...
F01287,Plant Taxonomy_Accepted species_TPL,NaN,Species epithet of plant according to The Plan...
F01289,Plant taxonomy_Accepted family_TPL,NaN,Family of plant according to The Plant List: T...
F01290,Plant taxonomy_Accepted order_APW,NaN,"Order of plant. For Angiosperms, this was dete..."


In [45]:
#---------------------------------------------------------------------------------------------------------------------------
# IGNORING SPELLING ERRORS IN THE BONIMONAL NAMES ALTOGETHER FOR THIS ANALYSIS
# PRIORITIZING ROOT TRAIT DATA AS IT CANNOT BE HARVESTED FROM ANYWHERE ELSE UNLIKE THE DISCRETE TRAITS
# FOR THIS, WE FOCUS ON GENERA INSTEAD OF SPECIES AS FUNGALROOT REQUIRES MANUAL DATA EXTRACTION FOR SPECIES LEVEL DATA
#---------------------------------------------------------------------------------------------------------------------------

### ___1. Records with data for RD & SRL___

In [52]:
# genera with complete binominal names and collaboration axis trait data (RD & SRL)
# we look for records that have data for both RD and SRL TOGETHER!!!!!
# root order not accounted for

binom_collab_genera = fred.dropna(subset=BINOMINAL + COLLABORATION_AXES).loc[:, "F01286"].str.strip().unique()
binom_collab_genera.size

532

In [53]:
states = pd.Series(index=binom_collab_genera, data=[fungalroot_genus_level_states.get(sp, np.nan) for sp in binom_collab_genera])
states.value_counts(dropna=False)

AM                                             438
NM-AM                                           33
EcM                                             27
NM                                              12
EcM-AM                                           7
ErM                                              6
NaN                                              5
uncertain                                        2
OM                                               1
species-specific: AM or rarely EcM-AM or AM      1
Name: count, dtype: int64

In [63]:
# families of the above records
lookup.query(r"index.isin(@binom_collab_genera)").family.sort_values().unique()

array(['Acanthaceae', 'Actinidiaceae', 'Adoxaceae', 'Altingiaceae',
       'Amaranthaceae', 'Amaryllidaceae', 'Anacardiaceae', 'Annonaceae',
       'Apiaceae', 'Apocynaceae', 'Aquifoliaceae', 'Araliaceae',
       'Araucariaceae', 'Arecaceae', 'Asparagaceae', 'Aspleniaceae',
       'Asteliaceae', 'Asteraceae', 'Atherospermataceae', 'Athyriaceae',
       'Berberidaceae', 'Betulaceae', 'Bignoniaceae', 'Blechnaceae',
       'Boraginaceae', 'Brassicaceae', 'Burseraceae', 'Campanulaceae',
       'Cannabaceae', 'Caprifoliaceae', 'Caricaceae', 'Caryophyllaceae',
       'Celastraceae', 'Cercidiphyllaceae', 'Chloranthaceae',
       'Cibotiaceae', 'Cistaceae', 'Clusiaceae', 'Combretaceae',
       'Convolvulaceae', 'Coriariaceae', 'Cornaceae', 'Corynocarpaceae',
       'Cunoniaceae', 'Cupressaceae', 'Cyatheaceae', 'Cyperaceae',
       'Cystopteridaceae', 'Daphniphyllaceae', 'Dennstaedtiaceae',
       'Dicksoniaceae', 'Dioscoreaceae', 'Dipterocarpaceae',
       'Dryopteridaceae', 'Ebenaceae', 'Elae

### ___2. Fine root records with data for RD & SRL___

In [55]:
# look for records with data for binominal names, SRL, RD and root order
# and the root order must be in range [1, 3]

binom_collab_fine_root_genera = fred.dropna(subset=BINOMINAL + COLLABORATION_AXES).query(r"F00056.isin((1, 2, 3))").loc[:, "F01286"].str.strip().unique()
binom_collab_fine_root_genera.size

211

In [56]:
states = pd.Series(index=binom_collab_fine_root_genera, data=[fungalroot_genus_level_states.get(sp, np.nan) for sp in binom_collab_fine_root_genera])
states.value_counts(dropna=False)

AM           176
EcM           16
NM-AM         10
EcM-AM         4
ErM            2
NM             1
NaN            1
uncertain      1
Name: count, dtype: int64

In [65]:
lookup.query(r"index.isin(@binom_collab_fine_root_genera)").family.sort_values().unique()

array(['Acanthaceae', 'Actinidiaceae', 'Adoxaceae', 'Altingiaceae',
       'Amaranthaceae', 'Anacardiaceae', 'Annonaceae', 'Apiaceae',
       'Apocynaceae', 'Aquifoliaceae', 'Araliaceae', 'Arecaceae',
       'Asparagaceae', 'Asteraceae', 'Athyriaceae', 'Berberidaceae',
       'Betulaceae', 'Bignoniaceae', 'Blechnaceae', 'Boraginaceae',
       'Burseraceae', 'Cannabaceae', 'Caprifoliaceae', 'Celastraceae',
       'Cercidiphyllaceae', 'Chloranthaceae', 'Cibotiaceae', 'Clusiaceae',
       'Coriariaceae', 'Cornaceae', 'Cupressaceae', 'Cystopteridaceae',
       'Daphniphyllaceae', 'Dennstaedtiaceae', 'Dioscoreaceae',
       'Dryopteridaceae', 'Ebenaceae', 'Elaeocarpaceae', 'Equisetaceae',
       'Ericaceae', 'Euphorbiaceae', 'Eupteleaceae', 'Fabaceae',
       'Fagaceae', 'Ginkgoaceae', 'Gleicheniaceae', 'Hamamelidaceae',
       'Hydrangeaceae', 'Hypericaceae', 'Iridaceae', 'Iteaceae',
       'Juglandaceae', 'Lamiaceae', 'Lauraceae', 'Lecythidaceae',
       'Magnoliaceae', 'Malvaceae', 'Mela

### ___3. First order root records with data for RD & SRL___

In [66]:
# look for records with data for binominal names, SRL, RD and root order
# and the root order must be 1

binom_collab_first_root_genera = fred.dropna(subset=BINOMINAL + COLLABORATION_AXES).query(r"F00056 == 1").loc[:, "F01286"].str.strip().unique()
binom_collab_first_root_genera.size

210

In [67]:
states = pd.Series(index=binom_collab_first_root_genera, data=[fungalroot_genus_level_states.get(sp, np.nan) for sp in binom_collab_first_root_genera])
states.value_counts(dropna=False)

AM           175
EcM           16
NM-AM         10
EcM-AM         4
ErM            2
NM             1
NaN            1
uncertain      1
Name: count, dtype: int64

In [68]:
lookup.query(r"index.isin(@binom_collab_first_root_genera)").family.sort_values().unique()

array(['Acanthaceae', 'Actinidiaceae', 'Adoxaceae', 'Altingiaceae',
       'Amaranthaceae', 'Anacardiaceae', 'Annonaceae', 'Apiaceae',
       'Apocynaceae', 'Aquifoliaceae', 'Araliaceae', 'Arecaceae',
       'Asparagaceae', 'Asteraceae', 'Athyriaceae', 'Berberidaceae',
       'Betulaceae', 'Bignoniaceae', 'Blechnaceae', 'Boraginaceae',
       'Burseraceae', 'Cannabaceae', 'Caprifoliaceae', 'Celastraceae',
       'Cercidiphyllaceae', 'Chloranthaceae', 'Cibotiaceae', 'Clusiaceae',
       'Coriariaceae', 'Cornaceae', 'Cupressaceae', 'Cystopteridaceae',
       'Daphniphyllaceae', 'Dennstaedtiaceae', 'Dioscoreaceae',
       'Dryopteridaceae', 'Ebenaceae', 'Elaeocarpaceae', 'Equisetaceae',
       'Ericaceae', 'Euphorbiaceae', 'Eupteleaceae', 'Fabaceae',
       'Fagaceae', 'Ginkgoaceae', 'Gleicheniaceae', 'Hamamelidaceae',
       'Hydrangeaceae', 'Hypericaceae', 'Iridaceae', 'Iteaceae',
       'Juglandaceae', 'Lamiaceae', 'Lauraceae', 'Lecythidaceae',
       'Magnoliaceae', 'Malvaceae', 'Mela

In [81]:
#--------------------------------------------------------------
# LOOK FOR EACH COLLABORATION AXIS TRAIT SEPARATELY
#--------------------------------------------------------------

### ___4. Records with RD data___

In [94]:
genera = fred.dropna(subset=BINOMINAL + ["F00679"]).loc[:, "F01286"].str.strip().unique()
states = pd.Series(index=genera, data=[fungalroot_genus_level_states.get(sp, np.nan) for sp in genera])
states.value_counts(dropna=False)

AM                                             512
NM-AM                                           53
EcM                                             27
NM                                              14
ErM                                             11
EcM-AM                                           7
NaN                                              7
uncertain                                        3
OM                                               1
species-specific: AM or rarely EcM-AM or AM      1
Name: count, dtype: int64

In [95]:
lookup.query(r"index.isin(@genera)").family.sort_values().unique()

array(['Acanthaceae', 'Actinidiaceae', 'Adoxaceae', 'Alismataceae',
       'Altingiaceae', 'Amaranthaceae', 'Amaryllidaceae', 'Anacardiaceae',
       'Annonaceae', 'Apiaceae', 'Apocynaceae', 'Aquifoliaceae',
       'Araliaceae', 'Araucariaceae', 'Arecaceae', 'Asparagaceae',
       'Aspleniaceae', 'Asteliaceae', 'Asteraceae', 'Atherospermataceae',
       'Athyriaceae', 'Berberidaceae', 'Betulaceae', 'Bignoniaceae',
       'Blechnaceae', 'Boraginaceae', 'Brassicaceae', 'Bromeliaceae',
       'Burseraceae', 'Campanulaceae', 'Cannabaceae', 'Caprifoliaceae',
       'Caricaceae', 'Caryophyllaceae', 'Celastraceae',
       'Cercidiphyllaceae', 'Chloranthaceae', 'Cibotiaceae', 'Cistaceae',
       'Clusiaceae', 'Combretaceae', 'Commelinaceae', 'Convolvulaceae',
       'Coriariaceae', 'Cornaceae', 'Corynocarpaceae', 'Cucurbitaceae',
       'Cunoniaceae', 'Cupressaceae', 'Cyatheaceae', 'Cyperaceae',
       'Cystopteridaceae', 'Daphniphyllaceae', 'Dennstaedtiaceae',
       'Dicksoniaceae', 'Dioscor

### ___5. Fine root records with RD data___

In [96]:
genera = fred.dropna(subset=BINOMINAL + ["F00679"]).query(r"F00056.isin((1, 2, 3))").loc[:, "F01286"].str.strip().unique()
states = pd.Series(index=genera, data=[fungalroot_genus_level_states.get(sp, np.nan) for sp in genera])
states.value_counts(dropna=False)

AM           250
NM-AM         20
EcM           17
EcM-AM         4
ErM            3
NM             3
NaN            2
uncertain      1
Name: count, dtype: int64

In [97]:
lookup.query(r"index.isin(@genera)").family.sort_values().unique()

array(['Acanthaceae', 'Actinidiaceae', 'Adoxaceae', 'Altingiaceae',
       'Amaranthaceae', 'Amaryllidaceae', 'Anacardiaceae', 'Annonaceae',
       'Apiaceae', 'Apocynaceae', 'Aquifoliaceae', 'Araliaceae',
       'Arecaceae', 'Asparagaceae', 'Asteraceae', 'Athyriaceae',
       'Berberidaceae', 'Betulaceae', 'Bignoniaceae', 'Blechnaceae',
       'Boraginaceae', 'Bromeliaceae', 'Burseraceae', 'Cannabaceae',
       'Caprifoliaceae', 'Celastraceae', 'Cercidiphyllaceae',
       'Chloranthaceae', 'Cibotiaceae', 'Clusiaceae', 'Coriariaceae',
       'Cornaceae', 'Cupressaceae', 'Cyperaceae', 'Cystopteridaceae',
       'Daphniphyllaceae', 'Dennstaedtiaceae', 'Dioscoreaceae',
       'Dryopteridaceae', 'Ebenaceae', 'Elaeagnaceae', 'Elaeocarpaceae',
       'Equisetaceae', 'Ericaceae', 'Euphorbiaceae', 'Eupteleaceae',
       'Fabaceae', 'Fagaceae', 'Ginkgoaceae', 'Gleicheniaceae',
       'Grossulariaceae', 'Hamamelidaceae', 'Hydrangeaceae',
       'Hypericaceae', 'Icacinaceae', 'Iridaceae', 'Iteace

### ___6. First order root records with RD data___

In [98]:
genera = fred.dropna(subset=BINOMINAL + ["F00679"]).query(r"F00056==1").loc[:, "F01286"].str.strip().unique()
states = pd.Series(index=genera, data=[fungalroot_genus_level_states.get(sp, np.nan) for sp in genera])
states.value_counts(dropna=False)

AM           249
NM-AM         20
EcM           17
EcM-AM         4
ErM            3
NM             3
NaN            2
uncertain      1
Name: count, dtype: int64

In [99]:
lookup.query(r"index.isin(@genera)").family.sort_values().unique()

array(['Acanthaceae', 'Actinidiaceae', 'Adoxaceae', 'Altingiaceae',
       'Amaranthaceae', 'Amaryllidaceae', 'Anacardiaceae', 'Annonaceae',
       'Apiaceae', 'Apocynaceae', 'Aquifoliaceae', 'Araliaceae',
       'Arecaceae', 'Asparagaceae', 'Asteraceae', 'Athyriaceae',
       'Berberidaceae', 'Betulaceae', 'Bignoniaceae', 'Blechnaceae',
       'Boraginaceae', 'Bromeliaceae', 'Burseraceae', 'Cannabaceae',
       'Caprifoliaceae', 'Celastraceae', 'Cercidiphyllaceae',
       'Chloranthaceae', 'Cibotiaceae', 'Clusiaceae', 'Coriariaceae',
       'Cornaceae', 'Cupressaceae', 'Cyperaceae', 'Cystopteridaceae',
       'Daphniphyllaceae', 'Dennstaedtiaceae', 'Dioscoreaceae',
       'Dryopteridaceae', 'Ebenaceae', 'Elaeagnaceae', 'Elaeocarpaceae',
       'Equisetaceae', 'Ericaceae', 'Euphorbiaceae', 'Eupteleaceae',
       'Fabaceae', 'Fagaceae', 'Ginkgoaceae', 'Gleicheniaceae',
       'Grossulariaceae', 'Hamamelidaceae', 'Hydrangeaceae',
       'Hypericaceae', 'Icacinaceae', 'Iridaceae', 'Iteace

### ___7. Records with SRL data___

In [100]:
genera = fred.dropna(subset=BINOMINAL + ["F00727"]).loc[:, "F01286"].str.strip().unique()
states = pd.Series(index=genera, data=[fungalroot_genus_level_states.get(sp, np.nan) for sp in genera])
states.value_counts(dropna=False)

AM                                             581
NM-AM                                           70
EcM                                             27
NM                                              21
NaN                                             12
EcM-AM                                           8
ErM                                              8
uncertain                                        3
OM                                               1
species-specific: AM or rarely EcM-AM or AM      1
Name: count, dtype: int64

In [101]:
lookup.query(r"index.isin(@genera)").family.sort_values().unique()

array(['Acanthaceae', 'Actinidiaceae', 'Adoxaceae', 'Altingiaceae',
       'Amaranthaceae', 'Amaryllidaceae', 'Anacardiaceae', 'Annonaceae',
       'Apiaceae', 'Apocynaceae', 'Aquifoliaceae', 'Araliaceae',
       'Araucariaceae', 'Arecaceae', 'Asparagaceae', 'Asphodelaceae',
       'Aspleniaceae', 'Asteliaceae', 'Asteraceae', 'Atherospermataceae',
       'Athyriaceae', 'Balsaminaceae', 'Berberidaceae', 'Betulaceae',
       'Bignoniaceae', 'Blechnaceae', 'Boraginaceae', 'Brassicaceae',
       'Burseraceae', 'Campanulaceae', 'Cannabaceae', 'Caprifoliaceae',
       'Caricaceae', 'Caryophyllaceae', 'Celastraceae',
       'Cercidiphyllaceae', 'Chloranthaceae', 'Cibotiaceae', 'Cistaceae',
       'Clusiaceae', 'Combretaceae', 'Convolvulaceae', 'Coriariaceae',
       'Cornaceae', 'Corynocarpaceae', 'Crassulaceae', 'Cunoniaceae',
       'Cupressaceae', 'Cyatheaceae', 'Cyperaceae', 'Cystopteridaceae',
       'Daphniphyllaceae', 'Dennstaedtiaceae', 'Dicksoniaceae',
       'Dioscoreaceae', 'Dipter

### ___8. Fine root records with SRL data___

In [102]:
genera = fred.dropna(subset=BINOMINAL + ["F00727"]).query(r"F00056.isin((1, 2, 3))").loc[:, "F01286"].str.strip().unique()
states = pd.Series(index=genera, data=[fungalroot_genus_level_states.get(sp, np.nan) for sp in genera])
states.value_counts(dropna=False)

AM           199
EcM           17
NM-AM         13
EcM-AM         4
ErM            2
NM             2
NaN            1
uncertain      1
Name: count, dtype: int64

In [103]:
lookup.query(r"index.isin(@genera)").family.sort_values().unique()

array(['Acanthaceae', 'Actinidiaceae', 'Adoxaceae', 'Altingiaceae',
       'Amaranthaceae', 'Amaryllidaceae', 'Anacardiaceae', 'Annonaceae',
       'Apiaceae', 'Apocynaceae', 'Aquifoliaceae', 'Araliaceae',
       'Arecaceae', 'Asparagaceae', 'Asphodelaceae', 'Asteraceae',
       'Athyriaceae', 'Berberidaceae', 'Betulaceae', 'Bignoniaceae',
       'Blechnaceae', 'Boraginaceae', 'Burseraceae', 'Cannabaceae',
       'Caprifoliaceae', 'Celastraceae', 'Cercidiphyllaceae',
       'Chloranthaceae', 'Cibotiaceae', 'Clusiaceae', 'Convolvulaceae',
       'Coriariaceae', 'Cornaceae', 'Crassulaceae', 'Cupressaceae',
       'Cyperaceae', 'Cystopteridaceae', 'Daphniphyllaceae',
       'Dennstaedtiaceae', 'Dioscoreaceae', 'Dryopteridaceae',
       'Ebenaceae', 'Elaeocarpaceae', 'Equisetaceae', 'Ericaceae',
       'Euphorbiaceae', 'Eupteleaceae', 'Fabaceae', 'Fagaceae',
       'Ginkgoaceae', 'Gleicheniaceae', 'Hamamelidaceae', 'Hydrangeaceae',
       'Hypericaceae', 'Iridaceae', 'Iteaceae', 'Juglandac

### ___9. First order root records with SRL data___

In [104]:
genera = fred.dropna(subset=BINOMINAL + ["F00727"]).query(r"F00056==1").loc[:, "F01286"].str.strip().unique()
states = pd.Series(index=genera, data=[fungalroot_genus_level_states.get(sp, np.nan) for sp in genera])
states.value_counts(dropna=False)

AM           198
EcM           16
NM-AM         13
EcM-AM         4
ErM            2
NM             2
NaN            1
uncertain      1
Name: count, dtype: int64

In [105]:
lookup.query(r"index.isin(@genera)").family.sort_values().unique()

array(['Acanthaceae', 'Actinidiaceae', 'Adoxaceae', 'Altingiaceae',
       'Amaranthaceae', 'Amaryllidaceae', 'Anacardiaceae', 'Annonaceae',
       'Apiaceae', 'Apocynaceae', 'Aquifoliaceae', 'Araliaceae',
       'Arecaceae', 'Asparagaceae', 'Asphodelaceae', 'Asteraceae',
       'Athyriaceae', 'Berberidaceae', 'Betulaceae', 'Bignoniaceae',
       'Blechnaceae', 'Boraginaceae', 'Burseraceae', 'Cannabaceae',
       'Caprifoliaceae', 'Celastraceae', 'Cercidiphyllaceae',
       'Chloranthaceae', 'Cibotiaceae', 'Clusiaceae', 'Convolvulaceae',
       'Coriariaceae', 'Cornaceae', 'Crassulaceae', 'Cupressaceae',
       'Cyperaceae', 'Cystopteridaceae', 'Daphniphyllaceae',
       'Dennstaedtiaceae', 'Dioscoreaceae', 'Dryopteridaceae',
       'Ebenaceae', 'Elaeocarpaceae', 'Equisetaceae', 'Ericaceae',
       'Euphorbiaceae', 'Eupteleaceae', 'Fabaceae', 'Fagaceae',
       'Ginkgoaceae', 'Gleicheniaceae', 'Hamamelidaceae', 'Hydrangeaceae',
       'Hypericaceae', 'Iridaceae', 'Iteaceae', 'Juglandac